### Abstract
This paper uses three supervised machine learning models, Logistic Regression, Random Forest, and XGBoost, to predict flight disruptions, which are cancellations, delays of 15 or more minutes, and delay duration. Using U.S. flight data, we evaluate models on recall, precision, F1-score, and RMSE. Our decision tree models outperform our linear ones in classification recall but suffer from low precision. None outperformed the baseline in predicting delay duration, highlighting the limits of our features. Key predictors include scheduled hours and airline type. We conclude that more operational data is needed to improve model accuracy and provide real value to our readers.


### Introduction


Disruptions, whether a delayed or canceled flight can can cause significant operational and economic stress on both airlines and passengers. These problems can lead to schedule changes and revenue loss for airlines, along with missed connections and disrupted travel plans for customers. Since the pandemic, air travel has boomed, leading to these issues becoming an even larger issue for airlines and passengers. As a result, being able to predict these operation delays has become crucial for both the business and the customers.

In this paper, we explore the use of supervised machine learning to predict three types of flight disruption outcomes: whether a flight will be cancelled, whether a flight will be delayed with the minimum length for a delay being 15 minutes or more, and the estimated length of a delay. These tasks are treated as a binary classification problem and a regression problem, respectively. We use a variety of models, which are Logistic Regression, Random Forest, and XGBoost, and assess their performance using standard evaluation metrics. Given the high class imbalance in cancellation and delay data, we pay particular attention to recall, precision, and F1-score rather than relying solely on accuracy.

This research contributes to the growing literature on predictive modeling in transportation, particularly aviation, by evaluating how different algorithms handle structured, tabular flight data and identifying which variables provide the most predictive power. In particular, we aim to assess whether our decision tree algorithms meaningfully outperform simpler linear models or basic baseline models, and to what extent the choice of features matters when predicting rare but impactful events like cancellations or long delays.


# Motivation/State of the Art


### Why Do This
Flight disruptions which can cancellations or delays cost the U.S. economy billions annually, cause significant passenger inconvenience, and put unnesecary stress on airline operations throughout the country. Predicting these disruptions in advance can help airlines to make better routing and crew decisions, help airports allocate gate and runway resources more effectively, and allow passengers to make more informed travel plans. With the rise of the rise of more informative data about air travel disruption, improved machine learning algorithims, we want to answer the following questions. Can we use Machine Learning to help solve predict these problems. If we can do this what are the results and what can we say aboit the results.


### Who Else Has Done This and Their Results



Predicting flight delays and cancellations has become a big topic in both academic and real world spaces. With more access to large flight datasets from government sources to airport level operational data researchers have tested a bunch of different machine learning models to try and predict disruptions. The goal in most of these papers is to reduce costs, help airlines run smoothly, and make the passenger experience better. This section highlights a few key studies that helped shape how we think about our own modeling and feature choices.

Hatipoğlu and Tosun (2024) focused on delay classification at a major Turkish airport. They tested a wide range of models and used resampling to handle the imbalance between delayed and non delayed flights. Their results showed that tree based models performed better than the simpler ones. Interestingly, timing related features like time of day and day of the week were much more useful for predictions than weather variables. That helped confirm our idea to lean more into time and airline features.


Rebollo and Balakrishnan (2014) looked at delays across the entire US airspace. They focused more on the system level than the individual flight level, using information about traffic flow and how delays spread across the network. Their models were able to predict major delays fairly well, especially when looking further out in time. While our work is more focused on the flight level, their results helped us think about the broader context of what causes delays.

Mtimkulu et al. (2023) did a delay prediction study at a South African airport. They tried out a few different models and showed that cleaning the data, selecting the right features, and balancing the classes made a huge difference. Like the others, they found that time and airline features were more useful than weather unless weather data was deeply tied to operational conditions. Their work helped support our decision to focus more on the structure of the flight data and less on external inputs we didn’t have strong coverage for.

Across all these studies, a pattern comes through tree based models like XGBoost and Random Forest tend to do better than linear ones. They also all stress the importance of dealing with imbalanced data and picking good features, especially around flight timing and airline operations. A lot of past papers treated delay and cancellation as one combined prediction problem, but we decided to split ours into three: cancellation, delay classification, and delay duration. That gives us more targeted models, helps us avoid leaking information across tasks, and makes it easier to see what really drives different kinds of disruptions.



# Data


### Original
Our data was found on Kaggle.com on a dataste titled "2015 Flight Delays and Cancellations". As described on the webpage the data was collected by by the DOT's Bureau of Transportation Statistics. The original dataset had 31 features and roughly 5.82 million obersvations. The original datasets featured can be descibed as the following detailing flight schedules, delays, cancellations, and other operational metrics. Here is an exact list of all the original features.


In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/Colab Notebooks/Final Paper'
original_flight_data = file_path + 'flights.csv'
df = pd.read_csv(original_flight_data)
print(f"Number of observations in dataframe: {len(df)}")
print("Columns in dataframe:")
for col in df.columns:
    print(col)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<ipython-input-102-634590400>:7: DtypeWarning: Columns (7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(original_flight_data)


Number of observations in dataframe: 5819079
Columns in dataframe:
YEAR
MONTH
DAY
DAY_OF_WEEK
AIRLINE
FLIGHT_NUMBER
TAIL_NUMBER
ORIGIN_AIRPORT
DESTINATION_AIRPORT
SCHEDULED_DEPARTURE
DEPARTURE_TIME
DEPARTURE_DELAY
TAXI_OUT
WHEELS_OFF
SCHEDULED_TIME
ELAPSED_TIME
AIR_TIME
DISTANCE
WHEELS_ON
TAXI_IN
SCHEDULED_ARRIVAL
ARRIVAL_TIME
ARRIVAL_DELAY
DIVERTED
CANCELLED
CANCELLATION_REASON
AIR_SYSTEM_DELAY
SECURITY_DELAY
AIRLINE_DELAY
LATE_AIRCRAFT_DELAY
WEATHER_DELAY


Before modeling, we performed essential data cleaning steps to ensure the dataset was usable for our specific machine learning problems. We began by dropping irrelevant categorical columns such as tail number and year. The tail number is similar to a car's license plate; it uniquely identifies each aircraft and indicates where it's registered. However, this information is not helpful for our predictive goals and introduces unnecessary information into the data. As for the year, it was constant across all observations, 2,015, making it irrelevant for model training. We also dropped the origin airport and the destination airport. While these features could theoretically provide valuable context, they introduced too much noise. When we hot encoded them, we ended up with over 660 additional columns. Reducing dimensionality by grouping airports by size or region was considered, but we determined it would still inject too much noise and potentially harm model performance. This would have also been by hand and would have introduced too much human bias into the model for our liking. For these reasons, we chose to exclude these variables from our datasets.

Following this, we did some feature engineering to make our dataset compatible with Machine Learning. This included making binary variables for each airline that was in the dataset. We also dropped features that would have introduced data leakage.. That is, information that would not be available at the time of prediction but is strongly correlated with the dependent variable. Examples include delay leaving the gate, time of arrival, time to taxi, time of actual take off, and amount of time in the air, which are only known after the flight has occurred. Including these features would inflate model performance while failing to simulate a real world predictive setting. Therefore, we excluded them from all datasets to ensure we did not have data leakage in our model. To reduce the number of features in our model, we grouped individual airlines by their operational type: Major, Low Cost, and Regional. This grouping allows us to retain the core operational differences between airlines, since those within the same group tend to operate similarly, while minimizing dimensionality. As a result, we capture the same predictive power with fewer features, thus creating less noise in the models.

Following that, we split our cleaned dataset into three separate subsets, each made to answer one of our research questions. The first subset focuses on predicting flight cancellations, using features available prior to departure, such as the airline, time of year, and scheduled departure time. The second subset is built to classify whether a flight will be delayed by more than 15 minutes, again using only pre-flight information to avoid data leakage. The third dataset filters the data to only include flights that were delayed and aims to predict the duration of the delay as a regression problem. The exact features and total number of observations can be seen below




In [ ]:
file_one = file_path + 'delay_cause_modeling_dataset.csv'
file_two = file_path + 'cancellation_prediction_dataset.csv'
file_three = file_path + 'delay_regression_dataset.csv'
df_cause = pd.read_csv(file_one)
df_cancel = pd.read_csv(file_two)
df_delay_reg = pd.read_csv(file_three)
datasets = ['Delay Cause Modeling Dataset',
    'Cancellation Prediction Dataset',
    'Delay Regression Dataset']
df_cancel = df_cancel.drop(columns=['DIVERTED'])
df_cause = df_cause.drop(columns=['DIVERTED'])
different_dataframes = [df_cause, df_cancel, df_delay_reg]
for i, df in enumerate(different_dataframes):
    print(f"The name of the is {datasets[i]}")
    print(f"Number of observations in dataframe: {len(df)}")
    print("Columns in dataframe:")
    for col in df.columns:
        print(col)



The name of the is Delay Cause Modeling Dataset
Number of observations in dataframe: 1153323
Columns in dataframe:
MONTH
DAY
DAY_OF_WEEK
SCHEDULED_DEPARTURE
DISTANCE
SCHEDULED_ARRIVAL
AIRLINE_Low Cost
AIRLINE_Major
AIRLINE_Regional
IS_WEEKEND
SCHEDULED_HOUR
CANCELLED_CARRIER
CANCELLED_WEATHER
CANCELLED_NATIONAL_AIR_SYSTEM
CANCELLED_SECURITY
The name of the is Cancellation Prediction Dataset
Number of observations in dataframe: 5819079
Columns in dataframe:
MONTH
DAY
DAY_OF_WEEK
SCHEDULED_DEPARTURE
DISTANCE
SCHEDULED_ARRIVAL
CANCELLED
AIRLINE_Low Cost
AIRLINE_Major
AIRLINE_Regional
IS_WEEKEND
SCHEDULED_HOUR
The name of the is Delay Regression Dataset
Number of observations in dataframe: 1023498
Columns in dataframe:
MONTH
DAY
DAY_OF_WEEK
SCHEDULED_DEPARTURE
DISTANCE
SCHEDULED_ARRIVAL
ARRIVAL_DELAY
DIVERTED
AIR_SYSTEM_DELAY
SECURITY_DELAY
AIRLINE_DELAY
LATE_AIRCRAFT_DELAY
WEATHER_DELAY
AIRLINE_Low Cost
AIRLINE_Major
AIRLINE_Regional
IS_WEEKEND
SCHEDULED_HOUR


To help address resource limitations, we will scale down the total size of the dataset by reducing the number of rows.

In [ ]:
df_cause = df_cause.sample(n=500000, random_state=42)
df_cancel = df_cancel.sample(n=500000, random_state=42)
df_delay_reg = df_delay_reg.sample(n=500000, random_state=42)


delay_cols = [
    'AIR_SYSTEM_DELAY',
    'SECURITY_DELAY',
    'AIRLINE_DELAY',
    'LATE_AIRCRAFT_DELAY',
    'WEATHER_DELAY'
]

for col in delay_cols:
    df_delay_reg[col] = df_delay_reg[col].apply(lambda x: 1 if x > 0 else 0)


### Class Inbalance
Another important part of the data preparation process is looking for a class imbalance within our dependent variables. The imbalance can negatively affect model training, as many algorithms tend to favor the majority class and may overlook rare but important events like cancellations or long delays.

In [ ]:
X_cancel = df_cancel.drop(columns=['CANCELLED'])
y_cancel = df_cancel['CANCELLED']


X_delay_reg = df_delay_reg.drop(columns=['ARRIVAL_DELAY'])
y_delay_reg = df_delay_reg['ARRIVAL_DELAY']


X_cause = df_cause.drop(columns=['CANCELLED_CARRIER', 'CANCELLED_WEATHER',
                                 'CANCELLED_NATIONAL_AIR_SYSTEM', 'CANCELLED_SECURITY'])
y_cause = df_cause[['CANCELLED_CARRIER', 'CANCELLED_WEATHER',
                    'CANCELLED_NATIONAL_AIR_SYSTEM', 'CANCELLED_SECURITY']]
print("Canceled Class Imbalance:")
print(y_cancel.value_counts())
print("By Percentage")
print(y_cancel.value_counts(normalize=True))


print("Cause Class Imbalance:")
print(y_cause.sum())
print("By Percentage")
print(y_cause.sum()/len(y_cause))


print("Check if we need to scale or normalzie data")
print(X_delay_reg.describe())




Canceled Class Imbalance:
CANCELLED
0    492260
1      7740
Name: count, dtype: int64
By Percentage
CANCELLED
0    0.98452
1    0.01548
Name: proportion, dtype: float64
Cause Class Imbalance:
CANCELLED_CARRIER                10918
CANCELLED_WEATHER                21371
CANCELLED_NATIONAL_AIR_SYSTEM     6881
CANCELLED_SECURITY                   9
dtype: int64
By Percentage
CANCELLED_CARRIER                0.021836
CANCELLED_WEATHER                0.042742
CANCELLED_NATIONAL_AIR_SYSTEM    0.013762
CANCELLED_SECURITY               0.000018
dtype: float64
Check if we need to scale or normalzie data
               MONTH            DAY    DAY_OF_WEEK  SCHEDULED_DEPARTURE  \
count  500000.000000  500000.000000  500000.000000        500000.000000   
mean        6.232074      15.561964       3.873416          1470.571408   
std         3.408560       8.794678       1.979226           454.417965   
min         1.000000       1.000000       1.000000             1.000000   
25%         3.000000   

 We see here in our cancellation prediction dataset, only about 1.5% of flights were actually canceled. That means if we just predicted not canceled every time, we’d still be right over 98% of the time. But obviously, that doesn’t help us actually identify which flights are at risk. We saw similar issues in the delay classification dataset about 17.6% of flights were delayed by more than 15 minutes. The delay cause modeling dataset had even more imbalance, with certain causes like security delays being almost nonexistent. These imbalances are important because they can make our models biased toward the majority class. To fix this, we explored different techniques like adjusting class weights or using oversampling methods to better train our models. We also see in our regression dataset that we need to do some scaling on the variables that have small values.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_delay_reg_scaled = scaler.fit_transform(X_delay_reg)


# Experiments

For our classification problems, we’re using Logistic Regression, XGBoost, and Random Forest. These are common machine learning models, but we’re not just picking them randomly we chose them because the research we read showed they work well for this kind of task. Hatıpoğlu and Tosun (2024) found that XGBoost did the best when predicting flight delays, with about 80% accuracy. Suryawanshi et al. (2021) also had strong results, getting around 88% accuracy using models like Random Forest and Gradient Boosting. Even Nikookar et al. (2020), working in a different country and setup, found that decision trees worked better than more complex models like neural networks. So based on what’s worked for others, we decided to stick with the models that have proven success.

### Baseline

For every experiment, it's important that we establish a baseline. This gives us something to compare our results to and helps us understand whether using machine learning actually improves predictions. Without a baseline, it’s hard to tell if our models are performing well or just okay. It also lets us see if the added complexity of using machine learning is worth it in a real world setting, or if a simple method would be just as effective. For Regression based we will use ZeroR. This is simple, for every observation it just out puts the mean of the dependent variable. We will also use this for our classifcation problems. Here it will just output the most frequent class.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error

zero_r_classifier = DummyClassifier(strategy='most_frequent')
zero_r_classifier.fit(X_cancel, y_cancel)

zero_r_regressor = DummyRegressor(strategy='mean')
zero_r_regressor.fit(X_delay_reg, y_delay_reg)

zero_r_regressor_predictions = zero_r_regressor.predict(X_delay_reg)
zero_r_classifier_predictions = zero_r_classifier.predict(X_cancel)

zero_r_regressor_mse = mean_squared_error(y_delay_reg, zero_r_regressor_predictions)
zero_r_regressor_rmse = np.sqrt(zero_r_regressor_mse)
zero_r_classifier_accuracy = accuracy_score(y_cancel, zero_r_classifier_predictions)


zero_r_delay_classifier = DummyClassifier(strategy='most_frequent')
zero_r_delay_classifier.fit(X_delay_class, y_delay_class)
delay_class_predictions = zero_r_delay_classifier.predict(X_delay_class)
delay_class_accuracy = accuracy_score(y_delay_class, delay_class_predictions)

zero_r_classifier_pct = zero_r_classifier_accuracy * 100
delay_class_pct = delay_class_accuracy * 100
print(f"ZeroR Regression MSE: {zero_r_regressor_rmse:.2f}")
print(f"ZeroR Cancellation Classification Accuracy: {zero_r_classifier_pct:.2f}%")
print(f"ZeroR Delay Classification Accuracy: {delay_class_pct:.2f}%")


ZeroR Regression MSE: 65.21
ZeroR Cancellation Classification Accuracy: 98.45%
ZeroR Delay Classification Accuracy: 82.37%


Now that we have our baselines, we can proceed with our actual experiments. First, we need to split our data into training and testing sets to evaluate performance properly. This allows us to simulate how the model might perform on unseen data. As well as seen in the papers we review in our motivation section we apply SMOTE to our unbalanced data to try and help with the class inbalance.

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE




X_cancel_train, X_cancel_test, y_cancel_train, y_cancel_test = train_test_split(X_cancel, y_cancel)
X_delay_class_train, X_delay_class_test, y_delay_class_train, y_delay_class_test = train_test_split(X_delay_class, y_delay_class)
X_delay_reg_train, X_delay_reg_test, y_delay_reg_train, y_delay_reg_test = train_test_split(X_delay_reg, y_delay_reg)
X_cause_train, X_cause_test, y_cause_train, y_cause_test = train_test_split(X_cause, y_cause)

smote = SMOTE(random_state=42)
X_cancel_train_res, y_cancel_train_res = smote.fit_resample(X_cancel_train, y_cancel_train)
X_delay_class_train_res, y_delay_class_train_res = smote.fit_resample(X_delay_class_train, y_delay_class_train)





 ### Cancellation Prediction

Here we will run our algorithims for our cancelation dataset. First we will run Logestic Regression to see what coeffieicents have significant prediction power. Following this we will drop the varialbes that are found to not be relevent to our problem. Following that we will rerun Logestic Regresssion as well as our other models. We will interpert the results in the next section



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier


cancel_lr = LogisticRegression(class_weight='balanced', max_iter=1000)
cancel_lr.fit(X_cancel_train, y_cancel_train)
cancel_lr_predictions = cancel_lr.predict(X_cancel_test)
cancel_lr_accuracy = accuracy_score(y_cancel_test, cancel_lr_predictions)
cancel_lr_report = classification_report(y_cancel_test, cancel_lr_predictions)
print(f"Logistic Regression Accuracy: {cancel_lr_accuracy * 100:.2f}%")
print("Logistic Regression Classification Report:")
print(cancel_lr_report)

print("Printing LR coeffiecinets Witht the name")
for feature, coef in zip(X_cancel_train.columns, cancel_lr.coef_[0]):
    print(f"{feature}: {coef}")



Logistic Regression Accuracy: 64.09%
Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.64      0.78    123031
           1       0.03      0.63      0.05      1969

    accuracy                           0.64    125000
   macro avg       0.51      0.63      0.42    125000
weighted avg       0.98      0.64      0.77    125000

Printing LR coeffiecinets Witht the name
MONTH: -0.1338575115584151
DAY: 0.0009316424965932817
DAY_OF_WEEK: -0.18477639790172112
SCHEDULED_DEPARTURE: -0.0008310997584256363
DISTANCE: -0.0002825541027338
SCHEDULED_ARRIVAL: 4.4646986912197424e-05
AIRLINE_Low Cost: -0.005633578581209708
AIRLINE_Major: 0.054672406073653396
AIRLINE_Regional: 0.8054305118399503
IS_WEEKEND: 0.615611168117933
SCHEDULED_HOUR: 0.10435507169710813


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


The linear regression results reveal that some features contribute meaningfully to the prediction, while others have minimal impact. Key predictors include AIRLINE_Regional, IS_WEEKEND, and AIRLINE_Low Cost, all of which show strong positive associations with the outcome. On the other hand, DAY_OF_WEEK, MONTH, and SCHEDULED_HOUR display notable negative effects, suggesting potential patterns related to scheduling and seasonality. To improve model simplicity and interpretability, we are removing variables with very small coefficients such as SCHEDULED_DEPARTURE, DISTANCE, SCHEDULED_ARRIVAL, DAY, and AIRLINE_Major as their influence is negligible and likely does not justify their inclusion in the model.

In [ ]:
from re import X
X_cancel_train = X_cancel_train.drop(columns=['SCHEDULED_DEPARTURE', 'DISTANCE', 'SCHEDULED_ARRIVAL', 'DAY', 'AIRLINE_Major'])
X_cancel_test = X_cancel_test.drop(columns=['SCHEDULED_DEPARTURE', 'DISTANCE', 'SCHEDULED_ARRIVAL', 'DAY', 'AIRLINE_Major'])

cancel_lr = LogisticRegression(class_weight='balanced', max_iter=1000)
cancel_lr.fit(X_cancel_train, y_cancel_train)
cancel_lr_predictions = cancel_lr.predict(X_cancel_test)
cancel_lr_accuracy = accuracy_score(y_cancel_test, cancel_lr_predictions)
cancel_lr_report = classification_report(y_cancel_test, cancel_lr_predictions)
print(f"\nLogistic Regression Accuracy: {cancel_lr_accuracy * 100:.2f}%")
print("Logistic Regression Classification Report:")
print(cancel_lr_report)


scale_pos_weight = len(y_cancel_train[y_cancel_train == 0]) / len(y_cancel_train[y_cancel_train == 1])
cancel_xgb = XGBClassifier(scale_pos_weight= scale_pos_weight,
                         random_state=42)
cancel_xgb.fit(X_cancel_train, y_cancel_train)
cancel_xgb_predictions = cancel_xgb.predict(X_cancel_test)
cancel_xgb_accuracy = accuracy_score(y_cancel_test, cancel_xgb_predictions)
cancel_xgb_report = classification_report(y_cancel_test, cancel_xgb_predictions)
print(f"\nXGBoost Accuracy: {cancel_xgb_accuracy * 100:.2f}%")
print("XGBoost Classification Report:")
print(cancel_xgb_report)

cancel_rf = RandomForestClassifier(class_weight='balanced',
                                  random_state=42)
cancel_rf.fit(X_cancel_train, y_cancel_train)
cancel_rf_predictions = cancel_rf.predict(X_cancel_test)
cancel_rf_accuracy = accuracy_score(y_cancel_test, cancel_rf_predictions)
cancel_rf_report = classification_report(y_cancel_test, cancel_rf_predictions)
print(f"\nRandom Forest Accuracy: {cancel_rf_accuracy * 100:.2f}%")
print("Random Forest Classification Report:")
print(cancel_rf_report)




Logistic Regression Accuracy: 64.12%
Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.64      0.78    123031
           1       0.03      0.62      0.05      1969

    accuracy                           0.64    125000
   macro avg       0.51      0.63      0.42    125000
weighted avg       0.98      0.64      0.77    125000


XGBoost Accuracy: 72.66%
XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.73      0.84    123031
           1       0.04      0.64      0.07      1969

    accuracy                           0.73    125000
   macro avg       0.51      0.69      0.45    125000
weighted avg       0.98      0.73      0.83    125000


Random Forest Accuracy: 77.20%
Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.78      0.87    123031
           1       0.04      0

### Delay Classification

Here we will run our algorithims for our Delay dataset. First we will run Logestic Regression to see what coeffieicents have significant prediction power. Following this we will drop the varialbes that are found to not be relevent to our problem. Following that we will rerun Logestic Regresssion as well as our other models. We will interpert the results in the next section



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

delay_lr = LogisticRegression(class_weight='balanced', max_iter=1000)
delay_lr.fit(X_delay_class_train, y_delay_class_train)
delay_lr_predictions = delay_lr.predict(X_delay_class_test)
delay_lr_accuracy = accuracy_score(y_delay_class_test, delay_lr_predictions)
delay_lr_report = classification_report(y_delay_class_test, delay_lr_predictions)
print(f"Logistic Regression Accuracy: {delay_lr_accuracy * 100:.2f}%")
print("Logistic Regression Classification Report:")
print(delay_lr_report)
print("Printing LR coeffiecinets Witht the name")
for feature, coef in zip(X_delay_class_train.columns, delay_lr.coef_[0]):
    print(f"{feature}: {coef}")


Logistic Regression Accuracy: 57.37%
Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.57      0.69    102759
           1       0.23      0.61      0.34     22241

    accuracy                           0.57    125000
   macro avg       0.55      0.59      0.51    125000
weighted avg       0.76      0.57      0.62    125000

Printing LR coeffiecinets Witht the name
MONTH: -0.029774500045098046
DAY: -0.000721721726923136
DAY_OF_WEEK: 0.010132565628176044
SCHEDULED_DEPARTURE: 0.0011694062634574974
DISTANCE: 5.6275830711706196e-05
SCHEDULED_ARRIVAL: 0.00020392634501468108
AIRLINE_Low Cost: 0.05235554517153148
AIRLINE_Major: -0.4207323514529969
AIRLINE_Regional: -0.2986537005503071
IS_WEEKEND: -0.18882042513311617
SCHEDULED_HOUR: -0.05447446342510531


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


The logistic regression results show that some features clearly matter more than others. AIRLINE_Major and AIRLINE_Regional have strong negative coefficients, meaning flights with those carriers are less likely to be on time. IS_WEEKEND and SCHEDULED_HOUR also reduce the likelihood of a positive outcome, which lines up with the idea that weekend and later flights are riskier. On the other hand, variables like DAY, DISTANCE, SCHEDULED_ARRIVAL, and SCHEDULED_DEPARTURE barely move the needle. Since they don't really influence the prediction, we're dropping them to keep the model clean and focused on what actually matters.

In [ ]:
X_delay_class_train = X_delay_class_train.drop(columns=['DAY','DISTANCE','SCHEDULED_ARRIVAL','SCHEDULED_DEPARTURE'])
X_delay_class_test = X_delay_class_test.drop(columns=['DAY','DISTANCE','SCHEDULED_ARRIVAL','SCHEDULED_DEPARTURE'])

delay_lr = LogisticRegression(class_weight='balanced', max_iter=1000)
delay_lr.fit(X_delay_class_train, y_delay_class_train)
delay_lr_predictions = delay_lr.predict(X_delay_class_test)
delay_lr_accuracy = accuracy_score(y_delay_class_test, delay_lr_predictions)
delay_lr_report = classification_report(y_delay_class_test, delay_lr_predictions)
print(f"Logistic Regression Accuracy: {delay_lr_accuracy * 100:.2f}%")
print("Logistic Regression Classification Report:")
print(delay_lr_report)

scale_pos_weight = len(y_delay_class_train[y_delay_class_train == 0]) / len(y_delay_class_train[y_delay_class_train == 1])
delay_xgb = XGBClassifier(scale_pos_weight= scale_pos_weight,
                         random_state=42)
delay_xgb.fit(X_delay_class_train, y_delay_class_train)
delay_xgb_predictions = delay_xgb.predict(X_delay_class_test)
delay_xgb_accuracy = accuracy_score(y_delay_class_test, delay_xgb_predictions)
delay_xgb_report = classification_report(y_delay_class_test, delay_xgb_predictions)
print(f"XGBoost Accuracy: {delay_xgb_accuracy * 100:.2f}%")
print("XGBoost Classification Report:")
print(delay_xgb_report)

delay_rf = RandomForestClassifier(class_weight='balanced',
                                  random_state=42)
delay_rf.fit(X_delay_class_train, y_delay_class_train)
delay_rf_predictions = delay_rf.predict(X_delay_class_test)
delay_rf_accuracy = accuracy_score(y_delay_class_test, delay_rf_predictions)
delay_rf_report = classification_report(y_delay_class_test, delay_rf_predictions)
print(f"Random Forest Accuracy: {delay_rf_accuracy * 100:.2f}%")
print("Random Forest Classification Report:")
print(delay_rf_report)



Logistic Regression Accuracy: 57.48%
Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.57      0.69    102759
           1       0.23      0.60      0.33     22241

    accuracy                           0.57    125000
   macro avg       0.55      0.58      0.51    125000
weighted avg       0.75      0.57      0.62    125000

XGBoost Accuracy: 58.38%
XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.57      0.69    102759
           1       0.24      0.63      0.35     22241

    accuracy                           0.58    125000
   macro avg       0.56      0.60      0.52    125000
weighted avg       0.76      0.58      0.63    125000

Random Forest Accuracy: 59.74%
Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.60      0.71    102759
           1       0.24      0.58

### Delay Regression

Here we will run our algorithims for our Delay Regression dataset. First we will run Linear Regression to see what coeffieicents have significant prediction power. Following this we will drop the varialbes that are found to not be relevent to our problem. Following that we will rerun Logestic Regresssion as well as our other models. We will interpert the results in the next section

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error



lr_delay_reg = LinearRegression()
lr_delay_reg.fit(X_delay_reg_train, y_delay_reg_train)
lr_delay_reg_predictions = lr_delay_reg.predict(X_delay_reg_test)
lr_delay_reg_mse = mean_squared_error(y_delay_reg_test, lr_delay_reg_predictions)
lr_delay_reg_mse = np.sqrt(lr_delay_reg_mse)
print(f"Linear Regression MSE: {lr_delay_reg_mse:.2f}")
print("Printing LR coeffiecinets Witht the name")
for feature, coef in zip(X_delay_reg.columns, lr_delay_reg.coef_):
    print(f"{feature}: {coef}")

Linear Regression MSE: 65.36
Printing LR coeffiecinets Witht the name
MONTH: 0.22518012099033
DAY: 0.07015901031074183
DAY_OF_WEEK: -1.3107458275705737
SCHEDULED_DEPARTURE: -0.028312484227612
DISTANCE: 0.001494890121566956
SCHEDULED_ARRIVAL: 0.0016283798589604977
DIVERTED: 3.552713678800501e-15
AIR_SYSTEM_DELAY: -1.6193145949806491
SECURITY_DELAY: -5.1436180157166955
AIRLINE_DELAY: 10.300499833898886
LATE_AIRCRAFT_DELAY: 12.53674983223343
WEATHER_DELAY: 35.670477787149935
AIRLINE_Low Cost: 4.269921608564656
AIRLINE_Major: -4.823232078514199
AIRLINE_Regional: 0.5533104699495213
IS_WEEKEND: 4.74097304605629
SCHEDULED_HOUR: 2.2061549132757445


We will now drop the variables that do not carry much prediction power. Following that we will rerun Linear Regression as well as the rest of our algorithims.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

X_delay_reg_train = X_delay_reg_train.drop(columns=['SCHEDULED_DEPARTURE','DISTANCE','SCHEDULED_ARRIVAL','DIVERTED','DAY','MONTH','AIRLINE_Regional','AIRLINE_Major','AIRLINE_Low Cost','SCHEDULED_HOUR'])
X_delay_reg_test = X_delay_reg_test.drop(columns=['SCHEDULED_DEPARTURE','DISTANCE','SCHEDULED_ARRIVAL','DIVERTED','DAY','MONTH','AIRLINE_Regional','AIRLINE_Major','AIRLINE_Low Cost','SCHEDULED_HOUR'])

lr_delay_reg_post_drop = LinearRegression()
lr_delay_reg_post_drop.fit(X_delay_reg_train, y_delay_reg_train)
lr_delay_reg_post_drop.predictions = lr_delay_reg_post_drop.predict(X_delay_reg_test)
lr_delay_reg_post_drop_mse = mean_squared_error(y_delay_reg_test, lr_delay_reg_post_drop.predictions)
lr_delay_reg_post_drop_mse = np.sqrt(lr_delay_reg_post_drop_mse)
print(f"\nLinear Regression MSE: {lr_delay_reg_post_drop_mse:.2f}")
print("Printing LR coeffiecinets Witht the name")
for feature, coef in zip(X_delay_reg_train.columns, lr_delay_reg_post_drop.coef_):
    print(f"{feature}: {coef}")

rf_delay_reg = RandomForestRegressor(random_state=42)
rf_delay_reg.fit(X_delay_reg_train, y_delay_reg_train)
rf_delay_reg.predictions = rf_delay_reg.predict(X_delay_reg_test)
rf_delay_reg_mse = mean_squared_error(y_delay_reg_test, rf_delay_reg.predictions)
rf_delay_reg_mse = np.sqrt(rf_delay_reg_mse)
print(f"\nRandom Forest MSE: {rf_delay_reg_mse:.2f}")
print("Printing RF coeffiecinets Witht the name")
for feature, coef in zip(X_delay_reg_train.columns, rf_delay_reg.feature_importances_):
    print(f"{feature}: {coef}")

xgboost_delay_reg = XGBRegressor(random_state=42)
xgboost_delay_reg.fit(X_delay_reg_train, y_delay_reg_train)
xgboost_delay_reg.predictions = xgboost_delay_reg.predict(X_delay_reg_test)
xgboost_delay_reg_mse = mean_squared_error(y_delay_reg_test, xgboost_delay_reg.predictions)
xgboost_delay_reg_mse = np.sqrt(xgboost_delay_reg_mse)
print(f"\nXGBoost MSE: {xgboost_delay_reg_mse:.2f}")
print("Printing XGBoost coeffiecinets Witht the name")
for feature, coef in zip(X_delay_reg_train.columns, xgboost_delay_reg.feature_importances_):
    print(f"{feature}: {coef}")




Linear Regression MSE: 65.48
Printing LR coeffiecinets Witht the name
DAY_OF_WEEK: -1.3748955637620954
AIR_SYSTEM_DELAY: -1.029168127674005
SECURITY_DELAY: -5.004832571096682
AIRLINE_DELAY: 9.771672840878828
LATE_AIRCRAFT_DELAY: 10.988891983364837
WEATHER_DELAY: 35.04461360972278
IS_WEEKEND: 5.059971177660504

Random Forest MSE: 65.26
Printing RF coeffiecinets Witht the name
DAY_OF_WEEK: 0.05223222810997145
AIR_SYSTEM_DELAY: 0.09410590285189258
SECURITY_DELAY: 0.003610717003828834
AIRLINE_DELAY: 0.2671614602190652
LATE_AIRCRAFT_DELAY: 0.20910728998412026
WEATHER_DELAY: 0.3704491006230595
IS_WEEKEND: 0.003333301208062169

XGBoost MSE: 65.27
Printing XGBoost coeffiecinets Witht the name
DAY_OF_WEEK: 0.015640905126929283
AIR_SYSTEM_DELAY: 0.06287577748298645
SECURITY_DELAY: 0.0026820856146514416
AIRLINE_DELAY: 0.18840056657791138
LATE_AIRCRAFT_DELAY: 0.14983588457107544
WEATHER_DELAY: 0.5805647373199463
IS_WEEKEND: 0.0


# Results

### Cancelation Prediction

Below, we highlight the most important findings from our cancellation prediction experiments, including baseline performance, model metrics, and key feature importance across all three models.

In [ ]:
print("Baseline ZeroR Accuracy")
print(f"ZeroR Cancellation Classification Accuracy: {zero_r_classifier_pct:.2f}%")
print("Logistic Regression Results")
print(f"\nLogistic Regression Accuracy: {cancel_lr_accuracy * 100:.2f}%")
print("Logistic Regression Classification Report:")
print(cancel_lr_report)
print("Logestic Regression Coefficients")
for feature, coef in zip(X_cancel_train.columns, cancel_lr.coef_[0]):
    print(f"{feature}: {coef}")
print("\nXGBoost Results")
print(f"XGBoost Accuracy: {cancel_xgb_accuracy * 100:.2f}%")
print("XGBoost Classification Report:")
print(cancel_xgb_report)
print("XGBoost Coefficients")
for feature, coef in zip(X_cancel_train.columns, cancel_xgb.feature_importances_):
    print(f"{feature}: {coef}")
print("\nRandom Forest Results")
print(f"Random Forest Accuracy: {cancel_rf_accuracy * 100:.2f}%")
print("Random Forest Classification Report:")
print(cancel_rf_report)
print("Random Forest Coefficients")
for feature, coef in zip(X_cancel_train.columns, cancel_rf.feature_importances_):
    print(f"{feature}: {coef}")

Baseline ZeroR Accuracy
ZeroR Cancellation Classification Accuracy: 98.45%
Logistic Regression Results

Logistic Regression Accuracy: 64.12%
Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.64      0.78    123031
           1       0.03      0.62      0.05      1969

    accuracy                           0.64    125000
   macro avg       0.51      0.63      0.42    125000
weighted avg       0.98      0.64      0.77    125000

Logestic Regression Coefficients
MONTH: -0.13348829479061514
DAY_OF_WEEK: -0.17623902659155022
AIRLINE_Low Cost: -0.0667179109885558
AIRLINE_Regional: 0.8753870890966916
IS_WEEKEND: 0.541191603597345
SCHEDULED_HOUR: 0.024448666765395127

XGBoost Results
XGBoost Accuracy: 72.66%
XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.73      0.84    123031
           1       0.04      0.64      0.07      1969

    accuracy   

The ZeroR baseline model, which always predicts the majority class in this case, non cancelled flights, achieves an accuracy of 98.45%, meaning that non cancelled flights represent 98.45% of of dataset. This tells us that we have a very unbalanced dataset. We now move on with our analysis of more complex models.

Logistic Regression improves the model’s ability to identify cancellations, reaching a recall of 62%. But with a precision of only 3%, the majority of its cancellation predictions are incorrect. Its overall accuracy drops to 64.12%, and the F1-score for cancelled flights is just 0.05, reflecting the poor balance between identifying cancellations and limiting false alarms, which was expected with such a low precision score. The coefficients suggest that cancellations are more likely for regional carriers and on weekends. This makes sense operationally regional airlines often have more demanding schedules and fly to more remote areas, leaving them more vulnerable to disruptions. Surprisingly, the model suggests flights by low cost carriers are slightly less likely to be cancelled, which goes against the public narrative. Later departure times also slightly increase the odds of cancellation, though the effect is small.

XGBoost outperforms Logistic Regression in both overall accuracy and recall, achieving 72.66% accuracy and 64% recall for cancellations. Precision remains low at 4%, but the F1-score improves slightly to 0.07. Feature importance rankings show that departure time, airline type, especially regional carriers, and month are key predictors. Unlike Logistic Regression, the weekend variable carries no weight in this model, suggesting that the tree based models interpret it as relevant than the linear model shown above.

Random Forest performs the best overall in terms of accuracy, reaching 77.20%, though it loses some recall power, dropping to 56% for the cancelled class. Precision is 4%, and the F1-score stays at 0.07, mirroring XGBoost. Scheduled hour is again the most important feature, followed by month and day of the week. As with XGBoost, the weekend indicator and low cost carrier variables provide little additional predictive value.

Taken together, these models show progress in identifying cancellations compared to the naive baseline, but they still struggle with precision, showing that they predict wrong in a practical way more than they predict right. The strongest and most consistent signals across all three models are scheduled departure hour and whether the flight is operated by a regional carrier. However, the limited precision gives us a clear takeaway, to truly improve performance, we’ll need access to better features, such as real time data like weather conditions, aircraft turnaround times, or routing and crew assignments would be of significant help to improving our model.


### Delay Prediction

Below, we highlight the most important findings from our Delay prediction experiments, including baseline performance, model metrics, and key feature importance across all three models.

In [ ]:
print("Baseline ZeroR Accuracy")
print(f"ZeroR Delay Classification Accuracy: {delay_class_pct:.2f}%")
print("\nLogistic Regression Results")
print(f"Logistic Regression Accuracy: {delay_lr_accuracy * 100:.2f}%")
print("Logistic Regression Classification Report:")
print(delay_lr_report)
print("Logestic Regression Coefficients")
for feature, coef in zip(X_delay_class_train.columns, delay_lr.coef_[0]):
    print(f"{feature}: {coef}")
print("\nXGBoost Results")
print(f"XGBoost Accuracy: {delay_xgb_accuracy * 100:.2f}%")
print("XGBoost Classification Report:")
print(delay_xgb_report)
print("XGBoost Coefficients")
for feature, coef in zip(X_delay_class_train.columns, delay_xgb.feature_importances_):
    print(f"{feature}: {coef}")
print("\nRandom Forest Results")
print(f"Random Forest Accuracy: {delay_rf_accuracy * 100:.2f}%")
print("Random Forest Classification Report:")
print(delay_rf_report)
print("Random Forest Coefficients")
for feature, coef in zip(X_delay_class_train.columns, delay_rf.feature_importances_):
    print(f"{feature}: {coef}")


Baseline ZeroR Accuracy
ZeroR Delay Classification Accuracy: 82.37%

Logistic Regression Results
Logistic Regression Accuracy: 57.48%
Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.57      0.69    102759
           1       0.23      0.60      0.33     22241

    accuracy                           0.57    125000
   macro avg       0.55      0.58      0.51    125000
weighted avg       0.75      0.57      0.62    125000

Logestic Regression Coefficients
MONTH: -0.030341359946569717
DAY_OF_WEEK: 0.0084058889208809
AIRLINE_Low Cost: 0.11217136003514205
AIRLINE_Major: -0.3674963207571413
AIRLINE_Regional: -0.27121907057549755
IS_WEEKEND: -0.1489719894629955
SCHEDULED_HOUR: 0.07684038747100437

XGBoost Results
XGBoost Accuracy: 58.38%
XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.57      0.69    102759
           1       0.24      0.63      0

The baseline ZeroR classifier, which was discussed in the previous subsection, always predicts the class that appears the most. In this case, that is non delayed flights, which is 82.37% of flights in this dataset. This means that ZeroR has an overall accuracy of 82.37%. Now moving on to the more complex models.

Logistic Regression offers some improvement over the baseline by achieving a recall of 63% for delayed flights. However, its precision remains low at 24%, indicating a large number of false positives. The model’s overall accuracy drops to 57.48%, and the F1-score for delayed flights is just 0.33. These results highlight the difficulty of predicting delays in an imbalanced dataset. The model’s coefficients reveal that flights operated by low-cost carriers and those scheduled later in the day are more likely to be delayed. This makes sense. Low cost airlines often operate on a point to point network model, which keeps aircraft in near constant use to maximize revenue. However, this efficiency comes at a cost, when delays occur, they tend to have a snow ball effect throughout the system because resources and scheduling buffers are thin. Similarly, the positive association with later scheduled hours likely reflects how small delays accumulate over the day, leading to higher disruption rates in the afternoon and evening with less time to make up time as the day comes to a close. In contrast, flights operated by major and regional carriers are associated with lower delay likelihood, possibly due to better scheduling practices and greater access to backup resources due as flights either take off or land at a hub airport through the hub and spoke model. Interestingly, the variable weekend also has a negative coefficient, suggesting delays are less likely on weekends. This is somewhat surprising, as one might expect greater airport congestion during weekend travel periods. It may be the case, however, that while passenger volumes increase, operational schedules are less dense or more padded, reducing the rate of delays.

The XGBoost model shows slightly better performance than Logistic Regression, with an accuracy of 58.38% and an F1-score of 0.35 for delays. It achieves a recall of 63% and a precision of 24%, indicating improved identification of delayed flights, though false positives remain an issue. The model ranks scheduled departure time as the most important feature, reinforcing the pattern that flights later in the day are more prone to delays due to cumulative operational disruptions. Interestingly, XGBoost associates major airlines with a higher likelihood of delay, which contrasts with Logistic Regression’s negative coefficient. This is surprising given that major carriers typically operate under a hub and spoke model designed to reduce delays. Additionally, the weekend variable is given little to no prediction power, suggesting that weekend travel does not significantly influence delay predictions in this model.

Random Forest delivers the best overall accuracy among the models at 59.74%, though performance on delay detection remains similar to the others. It achieves a recall of 58% and a precision of 24%, producing an F1-score of 0.34 for the delayed class. As with XGBoost, the time of the flight stands out as the most influential variable. Other features, such as month, day, week, and budget airlines, offer some contribution, while the weekend or no weekend and regional airlines show little to no effect.

Overall, all three models struggle with the trade off between recall and precision when predicting delays similar to our previous classification problem, though they perform meaningfully better than the baseline. The hour of the flight emerges as the most consistent and important predictor of delays across all models. This makes sense due to how delays and operational blips can build up on one another as the day goes on. The persistence of low precision across classifiers underscores the difficulty of modeling imbalanced outcomes and highlights the need for further model tuning, resampling strategies, or additional contextual variables such as weather, airport congestion, where the aircraft or crew has been previously in the day, or if you are in the airline's hub may prove to be highly influential in this model.





### Delay Regression

Below, we highlight the most important findings from our Delay Regression experiments, including baseline performance, model metrics, and key feature importance across all three models.

In [ ]:
print("Baseline ZeroR Accuracy")
print(f"ZeroR Delay Regression MSE: {zero_r_regressor_rmse:.2f}")
print("\nLinear Regression Results Pre Feature Dropping")
print(f"Linear Regression MSE: {lr_delay_reg_mse:.2f}")
print("\nPrinting LR coeffiecinets Witht the name")
for feature, coef in zip(X_delay_reg_train.columns, lr_delay_reg.coef_):
    print(f"{feature}: {coef}")
print("\nLinear Regression Results Post Feature Dropping")
print(f"Linear Regression MSE: {lr_delay_reg_post_drop_mse:.2f}")
print("\nLogestic Regression Coefficients")
for feature, coef in zip(X_delay_reg_train.columns, lr_delay_reg_post_drop.coef_):
    print(f"{feature}: {coef}")
print("\nXGBoost Results")
print(f"XGBoost MSE: {xgboost_delay_reg_mse:.2f}")
print("\nXGBoost Coefficients")
for feature, coef in zip(X_delay_reg_train.columns, xgboost_delay_reg.feature_importances_):
    print(f"{feature}: {coef}")
print("\nRandom Forest Results")
print(f"Random Forest MSE: {rf_delay_reg_mse:.2f}")
print("\nRandom Forest Coefficients")
for feature, coef in zip(X_delay_reg_train.columns, rf_delay_reg.feature_importances_):
    print(f"{feature}: {coef}")


Baseline ZeroR Accuracy
ZeroR Delay Regression MSE: 65.21

Linear Regression Results Pre Feature Dropping
Linear Regression MSE: 65.36

Printing LR coeffiecinets Witht the name
DAY_OF_WEEK: 0.22518012099033
AIR_SYSTEM_DELAY: 0.07015901031074183
SECURITY_DELAY: -1.3107458275705737
AIRLINE_DELAY: -0.028312484227612
LATE_AIRCRAFT_DELAY: 0.001494890121566956
WEATHER_DELAY: 0.0016283798589604977
IS_WEEKEND: 3.552713678800501e-15

Linear Regression Results Post Feature Dropping
Linear Regression MSE: 65.48

Logestic Regression Coefficients
DAY_OF_WEEK: -1.3748955637620954
AIR_SYSTEM_DELAY: -1.029168127674005
SECURITY_DELAY: -5.004832571096682
AIRLINE_DELAY: 9.771672840878828
LATE_AIRCRAFT_DELAY: 10.988891983364837
WEATHER_DELAY: 35.04461360972278
IS_WEEKEND: 5.059971177660504

XGBoost Results
XGBoost MSE: 65.27

XGBoost Coefficients
DAY_OF_WEEK: 0.015640905126929283
AIR_SYSTEM_DELAY: 0.06287577748298645
SECURITY_DELAY: 0.0026820856146514416
AIRLINE_DELAY: 0.18840056657791138
LATE_AIRCRAFT_DE

The baseline ZeroR model, which simply predicts the mean arrival delay for all flights for all observations in the dataset, produces an RMSE of 65.21. While basic, this baseline offers a useful benchmark, any predictive model must outperform it to be considered effective.

We first applied Linear Regression using the full set of features, resulting in an RMSE of 65.36, slightly worse than the baseline. This minimal decline in performance suggests that the model is unable to meaningfully extract predictive relationships from the data, likely due to weak feature relevance or multicollinearity. Most coefficients in this version were extremely small, indicating that individual features had little marginal effect on delay prediction. For instance, even known causes of delay, like airline related and late aircraft delays, had near zero weights, while security delays had an unexpectedly large negative coefficient.

After dropping features with negligible coefficients, the simplified Linear Regression model was retrained and had an RMSE of 65.48, slightly higher than before. While the reduced model offered improved interpretability, the loss of predictive performance, however small, confirms that linear regression remains inadequate for capturing the complexity of delay durations in this context.

Our tree based models showed modest improvements. XGBoost achieved an RMSE of 65.27, slightly outperforming both linear regression variants. It identified weather delay as the most important feature, followed by airline delay and late aircraft delay. These variables had higher prediction scores because of their strong nonlinear relationships with total delay time. Unlike linear regression, XGBoost can capture nonlinear relationships, which allows it to use these features more effectively. Notably, XGBoost gave no importance to is weekend, suggesting that weekend status alone does not meaningfully affect delay duration according to this model.

Random Forest performed nearly identically, achieving the lowest RMSE of 65.26. Feature importance rankings mirrored those of XGBoost, with weather delay, airline delay, and late aircraft delay as the features with he highest predictive power. Smaller contributions were made by air system delay, day of week, and security delay. Like XGBoost, Random Forest largely ignored the is weekend variable. The similarity in results between these two models likely reflects their shared ability to model non-linear patterns and feature interactions.

Despite these improvements, all models performed within a narrow RMSE range, less than 0.3 units from the baseline. This indicates that, while these methods are better equipped to extract structure from the data, the existing features do not provide enough information to substantially improve predictive performance. In particular, delay duration appears to be driven by real-time operational and environmental factors that are not captured in the current dataset.

In summary, while models like XGBoost and Random Forest slightly outperform linear regression, none of them manage to beat the baseline ZeroR model in terms of RMSE. This highlights a key limitation we think, that is the available features, while they do offer some predictive power, they do not provide enough power to reduce error. Still, the coefficients rankings from tree based models consistently point to weather, airline, and late aircraft delays as the most relevant factors for predicting delay duration. To build a model that can make more accurate predictions, future work will need to include more data, such as real time weather at both airports, airport congestion levels, aircraft tail number histories, and crew scheduling details. These types of variables are more likely to capture the operational dynamics that drive delay length.


# Final Thoughts

Our analysis shows that tree-based models like Random Forest and XGBoost consistently outperform linear models on classification tasks such as predicting cancellations and delays. These models also provide better insight into feature importance, consistently ranking weather delays, airline delays, late aircraft delays, and time of day as the most powerful variables. However, for predicting delay duration a regression task none of the regression models, including decision trees, outperformed the ZeroR baseline. This suggests that the current feature set lacks the necessary information to meaningfully reduce prediction error for this problem.

While classification models achieve decent recall, they continue to struggle with precision, reflecting the difficulty of predicting rare events in highly imbalanced datasets. Based on our dataset, we do not find strong evidence that these machine learning models can currently offer reliable, actionable predictions for airlines or passengers. However, past research shows that this problem can be tackled effectively with machine learning. To reach that point, future models will need better features, specifically features related to real time weather, operational factors, and other aviation specific features. These additions are likely essential for machine learning to become a useful tool in flight disruption prediction.


# Work Cited
Hatıpoğlu, Irmak, and Ömür Tosun. “Predictive modeling of flight delays at an airport using machine learning methods.” Applied Sciences, vol. 14, no. 13, 24 June 2024, p. 5472, https://doi.org/10.3390/app14135472.

Mtimkulu, Zimele, and Mfowabo Maphosa. “Flight delay prediction using machine learning: A comparative study of ensemble techniques.” International Conference on Artificial Intelligence and Its Applications, vol. 2023, 9 Nov. 2023, pp. 212–218, https://doi.org/10.59200/icarti.2023.030.

Rebollo, Juan Jose, and Hamsa Balakrishnan. “Characterization and prediction of air traffic delays.” Transportation Research Part C: Emerging Technologies, vol. 44, July 2014, pp. 231–241, https://doi.org/10.1016/j.trc.2014.04.007.

Mamdouh, Maged, et al. “A novel intelligent approach for flight delay prediction.” Journal of Big Data, vol. 10, no. 1, 21 Dec. 2023, https://doi.org/10.1186/s40537-023-00854-w.


